# AI Incidents Pipeline — Google Colab

Este notebook corre el pipeline completo de análisis de incidentes de IA en Google Colab.

**Pasos:**
1. Clonar el repositorio e instalar dependencias (solo la primera vez)
2. Ejecutar las celdas en orden de arriba a abajo

> **Tip:** Si querés usar GPU para el análisis de sentimiento (más rápido), andá a `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU` antes de empezar.

## 1. Setup — clonar repositorio e instalar dependencias

Ejecutá esta celda **una sola vez** al inicio de cada sesión de Colab.

In [ ]:
import os

REPO_URL = "https://github.com/karenrg/incidents_pipeline"
REPO_DIR = "incidents_pipeline"

# Clonar solo si no existe ya
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"El directorio '{REPO_DIR}' ya existe, saltando clone.")

%cd {REPO_DIR}

# Instalar dependencias (se excluye torch y pytest: Colab ya trae torch preinstalado)
!grep -vE '^(torch|pytest)==' requirements.txt > /tmp/requirements_colab.txt
!pip install -r /tmp/requirements_colab.txt -q

print("\nSetup completo.")

## 2. Verificar entorno

In [ ]:
import torch

device = "GPU" if torch.cuda.is_available() else "CPU"
print(f"Dispositivo disponible: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Sin GPU. El paso de sentimiento (transformer) tardará más en CPU (~5-10 min).")

## 3. Imports y configuración

In [ ]:
from pathlib import Path

import yaml

from src import configure_logging, set_global_seeds
from src.ingestion import load_and_validate
from src.preprocessing import preprocess
from src.nlp import process_text
from src.sentiment import run_sentiment
from src.analysis import run_analysis
from src.visualization import run_visualization

with open("config/params.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

configure_logging()
set_global_seeds(config["random_state"])

print("Configuración cargada. random_state:", config["random_state"])

## 4. Ingestión

In [ ]:
df = load_and_validate(config)
df.head()

## 5. Preprocesamiento

In [ ]:
df = preprocess(df, config)
df.head()

## 6. NLP (tokenización y lematización)

In [ ]:
df = process_text(df, config)
df[["tokens", "mental_health_flag"]].head()

## 7. Análisis de sentimiento

> Esta celda descarga el modelo `cardiffnlp/twitter-roberta-base-sentiment-latest` (~500 MB) la primera vez.

In [ ]:
df = run_sentiment(df, config)
df[["sentiment_score", "sentiment_label"]].head()

## 8. Guardar dataset procesado

In [ ]:
processed_path = Path(config["data"]["processed_path"])
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(processed_path)
print(f"Dataset guardado en: {processed_path}")

## 9. Análisis descriptivo

In [ ]:
metrics = run_analysis(df, config)

## 10. Visualización y reporte PDF

In [ ]:
report_path = run_visualization(df, metrics, config)
print(f"Reporte generado en: {report_path}")

## 11. Descargar outputs

Descargá el reporte PDF y las figuras generadas.

In [ ]:
from google.colab import files
import glob

# Descargar reporte PDF
files.download(str(report_path))

# Descargar métricas JSON
files.download("outputs/reports/metrics.json")

print("Pipeline completado.")